<a href="https://colab.research.google.com/github/snwCat/mi_hazi3/blob/main/mi_hazi3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Adatok beolvasása

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("corvinus_mi2026-g25.csv")

print("Első 5 sor:")
print(df.head())

print("Oszlopok:")
print(df.columns)

print("Adathalmaz mérete:")
print(df.shape)

print("Hiányzó értékek oszloponként:")
print(df.isna().sum())

Adatfeldolgozás:
A fontosabb numerikus oszlopokat számmá alakítjuk.

In [ ]:

df["Movie_Rating"] = pd.to_numeric(df["Movie_Rating"], errors="coerce")
df["No_of_Ratings"] = pd.to_numeric(df["No_of_Ratings"], errors="coerce")
df["ReleaseYear"] = pd.to_numeric(df["ReleaseYear"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print("Adattípusok átalakítás után:")
print(df.dtypes)

Rendezők elemzése

In [ ]:
rendezok_szama = df["Directed_By"].dropna().nunique()
filmek_rendezo_szerint = df.groupby("Directed_By")["title"].count()

print("Különböző rendezők száma:", rendezok_szama)
print("Átlagos filmszám rendezőnként:", filmek_rendezo_szerint.mean())
print("Medián filmszám rendezőnként:", filmek_rendezo_szerint.median())
print("Rendezőnkénti filmszám szórása:", filmek_rendezo_szerint.std())

print("Top 10 legtöbb filmmel rendelkező rendező:")
print(filmek_rendezo_szerint.sort_values(ascending=False).head(10))

Színészek elemzése

In [ ]:
szineszek = df["Starring"].dropna().str.split(",").explode().str.strip()
kulonbozo_szineszek_szama = szineszek.nunique()
legtobb_filmben_szereplo_szinesz = szineszek.value_counts().idxmax()
legtobb_filmben_szereplo_szinesz_db = szineszek.value_counts().max()

print("Különböző színészek száma:", kulonbozo_szineszek_szama)
print("Legtöbb filmben szereplő színész:", legtobb_filmben_szereplo_szinesz)
print("Ennyi filmben szerepel:", legtobb_filmben_szereplo_szinesz_db)

print("Top 10 legtöbb filmben szereplő színész:")
print(szineszek.value_counts().head(10))

Értékelések elemzése

In [ ]:
atlag_ertekeles = df["Movie_Rating"].mean()
szoras_ertekeles = df["Movie_Rating"].std()

print("Filmek átlagértékelése:", atlag_ertekeles)
print("Értékelések szórása:", szoras_ertekeles)

print("Értékelések alapstatisztikái:")
print(df["Movie_Rating"].describe())

Kapcsolat az értékelések száma és az értékelés között

In [ ]:
df_tiszta = df.dropna(subset=["Movie_Rating", "No_of_Ratings"]).copy()

korrelacio = df_tiszta["No_of_Ratings"].corr(df_tiszta["Movie_Rating"])

# Logaritmikus átalakítás, mert az értékelésszámok nagyon eltérő nagyságrendűek lehetnek.
df_tiszta["log_No_of_Ratings"] = np.log1p(df_tiszta["No_of_Ratings"])
log_korrelacio = df_tiszta["log_No_of_Ratings"].corr(df_tiszta["Movie_Rating"])

print("Korreláció az értékelésszám és értékelés között:", korrelacio)
print("Korreláció logaritmikus értékelésszámmal:", log_korrelacio)

print("Értelmezés:")
if abs(log_korrelacio) < 0.2:
    print("Az értékelések száma és maga az értékelés között gyenge kapcsolat van.")
elif abs(log_korrelacio) < 0.5:
    print("Az értékelések száma és maga az értékelés között közepes kapcsolat van.")
else:
    print("Az értékelések száma és maga az értékelés között erős kapcsolat van.")

In [ ]:
# 7. EGYSZERŰ MODELL: LINEÁRIS REGRESSZIÓ
print("\n--- EGYSZERŰ MODELL: LINEÁRIS REGRESSZIÓ ---")

try:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score, mean_squared_error

X = df_tiszta[["log_No_of_Ratings"]]
    y = df_tiszta["Movie_Rating"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    modell = LinearRegression()
    modell.fit(X_train, y_train)

    y_pred = modell.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print("R² érték:", r2)
    print("RMSE:", rmse)
    print("Regressziós együttható:", modell.coef_[0])
    print("Tengelymetszet:", modell.intercept_)

    print("\nModell értelmezése:")
    if r2 < 0.1:
        print("A modell nagyon kevés magyarázóerővel rendelkezik.")
    elif r2 < 0.3:
        print("A modell gyenge magyarázóerővel rendelkezik.")
    elif r2 < 0.6:
        print("A modell közepes magyarázóerővel rendelkezik.")
    else:
        print("A modell viszonylag erős magyarázóerővel rendelkezik.")
except ModuleNotFoundError:
    print("A scikit-learn nincs telepítve, ezért a regressziós modell nem futott le.")
    print("Telepítéshez futtasd:")
    print("pip install scikit-learn")